# LLaVA-1.6 custom inference engine: real-GPU benchmark

Runs the repo's hand-written inference engine (`src/inference.py`) on a real
GPU and records:

- time-to-first-token (prefill latency), decode tokens/sec, peak VRAM
- GPU telemetry (SM utilization, memory) sampled via NVML during the run,
  split into the prefill window vs the decode window
- sequential vs batched throughput for the same requests

**Runtime**: Colab free T4 (16GB). The model loads 4-bit quantized (NF4)
because fp16 weights alone are ~14GB; expect ~15-20 min end to end on the
first run (most of it model download). Runtime > Change runtime type > T4,
then Run all. Results are written to `gpu_run_results.json`.

In [ ]:
import torch

assert torch.cuda.is_available(), "Select a GPU runtime: Runtime > Change runtime type > T4"
print(torch.cuda.get_device_name(0))

In [ ]:
# Clone the repo and install pinned deps (Colab already ships torch + CUDA).
import os

if not os.path.exists("modern-vqa-extension"):
    os.system("git clone https://github.com/AdivA-24/modern-vqa-extension.git")
os.chdir("modern-vqa-extension")

import sys

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

os.system("pip install -q 'transformers>=4.41' 'accelerate>=0.25.0' 'bitsandbytes>=0.43.0' 'pynvml>=11.5.0' 'Pillow>=10.0.0'")

In [ ]:
# Load the model through the repo's engine, 4-bit quantized for the T4.
import time

from src.inference import LlavaInferenceEngine

t0 = time.perf_counter()
engine = LlavaInferenceEngine(load_in_4bit=True)
load_seconds = time.perf_counter() - t0
vram_after_load = torch.cuda.memory_allocated() / 1024**3
print(f"model loaded in {load_seconds:.1f}s, VRAM after load: {vram_after_load:.2f} GiB")

In [ ]:
# NVML telemetry sampler: a background thread recording SM utilization and
# memory every 50ms, so we can compare the prefill window to the decode window.
import threading

import pynvml

pynvml.nvmlInit()
_handle = pynvml.nvmlDeviceGetHandleByIndex(0)


class TelemetrySampler:
    def __init__(self, interval=0.05):
        self.interval = interval
        self.samples = []
        self._stop = threading.Event()
        self._thread = None

    def start(self):
        self.samples = []
        self._stop.clear()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()

    def _run(self):
        while not self._stop.is_set():
            util = pynvml.nvmlDeviceGetUtilizationRates(_handle)
            mem = pynvml.nvmlDeviceGetMemoryInfo(_handle)
            self.samples.append(
                {
                    "t": time.perf_counter(),
                    "sm_util": util.gpu,
                    "mem_util": util.memory,
                    "mem_used_gib": mem.used / 1024**3,
                }
            )
            time.sleep(self.interval)

    def stop(self):
        self._stop.set()
        self._thread.join()
        return self.samples

    @staticmethod
    def window_stats(samples, start, end, key):
        vals = [s[key] for s in samples if start <= s["t"] <= end]
        if not vals:
            return None
        return {"mean": sum(vals) / len(vals), "max": max(vals), "n": len(vals)}

In [ ]:
# Test images (stable COCO val2017 URLs) and questions.
import io

import requests
from PIL import Image

IMAGES = {
    "cats": "http://images.cocodataset.org/val2017/000000039769.jpg",
    "street": "http://images.cocodataset.org/val2017/000000000139.jpg",
}
images = {
    name: Image.open(io.BytesIO(requests.get(url, timeout=30).content)).convert("RGB")
    for name, url in IMAGES.items()
}
REQUESTS = [
    ("cats", "What are the animals doing, and what objects are near them?"),
    ("street", "Describe the scene and count the people you can see."),
    ("cats", "What colors are the two animals?"),
]

In [ ]:
# Sequential runs through the hand-written decode loop, with telemetry.
runs = []
for name, question in REQUESTS:
    sampler = TelemetrySampler()
    sampler.start()
    t_start = time.perf_counter()
    result = engine.greedy_decode(images[name], question, max_new_tokens=100)
    samples = sampler.stop()

    prefill_end = t_start + result.metrics.prefill_seconds
    decode_end = prefill_end + result.metrics.decode_seconds
    run = {
        "image": name,
        "question": question,
        "answer": result.text,
        "metrics": result.metrics.as_dict(),
        "telemetry": {
            "prefill_sm_util": TelemetrySampler.window_stats(samples, t_start, prefill_end, "sm_util"),
            "decode_sm_util": TelemetrySampler.window_stats(samples, prefill_end, decode_end, "sm_util"),
            "decode_mem_util": TelemetrySampler.window_stats(samples, prefill_end, decode_end, "mem_util"),
            "peak_mem_used_gib": max(s["mem_used_gib"] for s in samples) if samples else None,
        },
    }
    runs.append(run)
    m = result.metrics
    print(f"[{name}] TTFT {m.prefill_seconds:.2f}s | {m.decode_tokens_per_second:.1f} tok/s | {m.generated_tokens} tokens")
    print(f"    answer: {result.text[:120]}")

In [ ]:
# Batched run: the same requests in one left-padded generate() call.
# Static batching amortizes weight reads across requests; per-request latency
# rises but aggregate tokens/sec should beat the sequential loop.
batch_images = [images[name] for name, _ in REQUESTS]
batch_questions = [q for _, q in REQUESTS]

t_start = time.perf_counter()
batch_results = engine.generate_batch(batch_images, batch_questions, max_new_tokens=100)
batch_wall = time.perf_counter() - t_start

seq_wall = sum(r["metrics"]["prefill_seconds"] + r["metrics"]["decode_seconds"] for r in runs)
seq_tokens = sum(r["metrics"]["generated_tokens"] for r in runs)
batch_tokens = sum(r.metrics.generated_tokens for r in batch_results)
print(f"sequential: {seq_tokens} tokens in {seq_wall:.1f}s = {seq_tokens/seq_wall:.1f} tok/s aggregate")
print(f"batched:    {batch_tokens} tokens in {batch_wall:.1f}s = {batch_tokens/batch_wall:.1f} tok/s aggregate")

In [ ]:
# Persist everything for the README results table.
import json

results = {
    "gpu": torch.cuda.get_device_name(0),
    "quantization": "4-bit NF4 (bitsandbytes), fp16 compute",
    "model": engine.model_name,
    "load_seconds": round(load_seconds, 1),
    "vram_after_load_gib": round(vram_after_load, 2),
    "sequential_runs": runs,
    "batch": {
        "n_requests": len(REQUESTS),
        "wall_seconds": round(batch_wall, 2),
        "aggregate_tokens_per_second": round(batch_tokens / batch_wall, 2),
        "sequential_aggregate_tokens_per_second": round(seq_tokens / seq_wall, 2),
    },
}
with open("../gpu_run_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2)[:2000])

Download `gpu_run_results.json` (left sidebar > Files) and drop it into the
repo; the README results section is generated from it.